# ArcGIS Quick Integration Test (HTTP Only)

This notebook follows the same step-by-step flow, but uses standard Python HTTP calls (no ArcGIS Python API library).

1. Load `.env` variables
2. Build an authenticated HTTP session
3. Validate portal connection
4. Run a simple ImageServer test (no error handling)

In [1]:
import os
from pathlib import Path

import requests
from dotenv import load_dotenv

# Load environment values from repository root .env
repo_root = Path.cwd()
if not (repo_root / '.env').exists() and (repo_root.parent / '.env').exists():
    repo_root = repo_root.parent

env_path = repo_root / '.env'
load_dotenv(env_path)

portal_url = os.getenv('ARCGIS_PORTAL_URL', '').strip()
api_key = os.getenv('ARCGIS_API_KEY', '').strip()
image_server_url = 'https://mapas.pd.anatel.gov.br/image/rest/services/DEM/ImageServer'
token_referer = os.getenv('ARCGIS_TOKEN_REFERER', '').strip()

print(f'Loaded .env from: {env_path}')
print(f'ARCGIS_PORTAL_URL set: {portal_url}')
print('ARCGIS_API_KEY set : yes' if api_key else 'ARCGIS_API_KEY set : no')
print(f'ImageServer URL   : {image_server_url}')
print(f"ARCGIS_TOKEN_REFERER: {token_referer or '(not set)'}")

Loaded .env from: c:\GitHub\MSGraphTest\.env
ARCGIS_PORTAL_URL set: https://mapas.pd.anatel.gov.br/portal
ARCGIS_API_KEY set : yes
ImageServer URL   : https://mapas.pd.anatel.gov.br/image/rest/services/DEM/ImageServer
ARCGIS_TOKEN_REFERER: (not set)


## Test 1: Build HTTP Session

Create a reusable `requests.Session` for ArcGIS REST calls.

In [2]:
if not portal_url:
    raise ValueError('ARCGIS_PORTAL_URL is required in .env')
if not api_key:
    raise ValueError('ARCGIS_API_KEY is required in .env')

session = requests.Session()

print('HTTP session created')

HTTP session created


## Test 2: Validate Portal Connection

Call Portal REST `portals/self` with `apikey` and validate the JSON response.

In [3]:
portal_self_url = f"{portal_url.rstrip('/')}/sharing/rest/portals/self"
portal_params = {
    'f': 'pjson',
    'apikey': api_key,
}

portal_resp = session.get(portal_self_url, params=portal_params, timeout=30)
portal_data = portal_resp.json()

print(f"HTTP status: {portal_resp.status_code}")

# ArcGIS may return semantic errors in JSON even when HTTP status is 200.
if isinstance(portal_data, dict) and 'error' in portal_data:
    err = portal_data['error']
    code = err.get('code', 'unknown')
    message = err.get('message', 'Unknown ArcGIS error')
    details = err.get('details') or []
    details_text = ' | '.join(str(d) for d in details) if details else '(no details)'
    raise RuntimeError(
        f"ArcGIS portal validation failed (code={code}): {message}. Details: {details_text}"
    )

selected_auth_param = 'apikey'
print(f"Selected auth mode for next calls: {selected_auth_param}")
print(f"Portal name: {portal_data.get('name', 'unknown')}")
print(f"Portal id  : {portal_data.get('id', 'unknown')}")
print(f"URL key    : {portal_data.get('urlKey', 'unknown')}")

HTTP status: 200
Selected auth mode for next calls: apikey
Portal name: ArcGIS Enterprise
Portal id  : 0123456789ABCDEF
URL key    : unknown


## Test 3: ImageServer Metadata With Token Fallback

Try direct ImageServer metadata with `apikey`. If the service returns `499 Token Required`, perform Portal -> Server token exchange using `generateToken` and retry with `token`.

If token exchange is referer-restricted, the cell will try multiple referer candidates (including `ARCGIS_TOKEN_REFERER` if provided).

In [4]:
from urllib.parse import urlparse

base_headers = {'Referer': token_referer} if token_referer else {}

# 1) First try direct call with apikey
service_params = {
    'f': 'pjson',
    'apikey': api_key,
}

print('Trying ImageServer metadata with auth mode: apikey')
service_resp = session.get(image_server_url, params=service_params, headers=base_headers, timeout=30)
service_data = service_resp.json()
print(f'HTTP status: {service_resp.status_code}')

# 2) If service asks for token, exchange via generateToken and retry
if isinstance(service_data, dict) and 'error' in service_data and service_data['error'].get('code') == 499:
    print('ImageServer returned 499 Token Required. Trying Portal->Server token exchange...')

    server_root = image_server_url.split('/rest/')[0]
    gen_url = f"{portal_url.rstrip('/')}/sharing/rest/generateToken"

    # Build referer candidates to satisfy strict credential policies.
    portal_origin = f"{urlparse(portal_url).scheme}://{urlparse(portal_url).netloc}" if portal_url else ''
    referer_candidates = []
    if token_referer:
        referer_candidates.append(token_referer)
    if portal_origin:
        referer_candidates.append(portal_origin)
        referer_candidates.append(portal_origin + '/')

    # De-duplicate while preserving order
    referer_candidates = list(dict.fromkeys([r for r in referer_candidates if r]))
    if not referer_candidates:
        referer_candidates = ['']

    gen_data = None
    last_error = None
    for idx, referer in enumerate(referer_candidates, start=1):
        headers = {'Referer': referer} if referer else {}
        payload = {
            'f': 'json',
            'token': api_key,
            'serverUrl': server_root,
            'client': 'referer',
        }
        if referer:
            payload['referer'] = referer

        print(f"Trying generateToken with referer candidate #{idx}: {referer or '(none)'}")
        gen_resp = session.post(gen_url, data=payload, headers=headers, timeout=30)
        gen_data = gen_resp.json()

        if isinstance(gen_data, dict) and 'error' in gen_data:
            last_error = gen_data['error']
            print(f"generateToken error #{idx}: {last_error}")
            continue

        if isinstance(gen_data, dict) and gen_data.get('token'):
            break

    if not isinstance(gen_data, dict) or 'error' in gen_data or not gen_data.get('token'):
        raise RuntimeError(
            "Unable to acquire server token for ImageServer. "
            "Set ARCGIS_TOKEN_REFERER in .env to the exact allowed referer configured for the credential. "
            f"Last generateToken error: {last_error}"
        )

    server_token = gen_data['token']
    print('Server token acquired (value hidden). Retrying ImageServer metadata with token...')

    service_params = {
        'f': 'pjson',
        'token': server_token,
    }
    retry_headers = {'Referer': token_referer} if token_referer else {}
    service_resp = session.get(image_server_url, params=service_params, headers=retry_headers, timeout=30)
    service_data = service_resp.json()
    print(f'HTTP status after token exchange: {service_resp.status_code}')

print(service_data)

Trying ImageServer metadata with auth mode: apikey
HTTP status: 200
ImageServer returned 499 Token Required. Trying Portal->Server token exchange...
Trying generateToken with referer candidate #1: https://mapas.pd.anatel.gov.br
generateToken error #1: {'code': 498, 'message': 'Invalid token.', 'details': ['Error validating token: Invalid Referer.']}
Trying generateToken with referer candidate #2: https://mapas.pd.anatel.gov.br/
generateToken error #2: {'code': 498, 'message': 'Invalid token.', 'details': ['Error validating token: Invalid Referer.']}


RuntimeError: Unable to acquire server token for ImageServer. Set ARCGIS_TOKEN_REFERER in .env to the exact allowed referer configured for the credential. Last generateToken error: {'code': 498, 'message': 'Invalid token.', 'details': ['Error validating token: Invalid Referer.']}